# YOLOv3 DPU Benchmark
Measures inference speed and power on the KV260 **FPGA DPU (B512)**.

YOLOv3 is an **object detection** model (detects multiple objects + bounding boxes).
Input: 416x416 image. Output: 3 feature maps at different scales.

**Run from Jupyter on the KV260** (`http://192.168.68.60:9090/lab`)

In [ ]:
import os, time, numpy as np

POWER_SENSOR = "/sys/class/hwmon/hwmon2/power1_input"
N_WARMUP = 5
N_BENCHMARK = 100

# Smart path: check local models/ first, then pynq default
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("dpu_bench.ipynb"))
XMODEL_LOCAL = os.path.join(NOTEBOOK_DIR, "models", "tf_yolov3_voc.xmodel")
XMODEL_PYNQ  = "/root/jupyter_notebooks/pynq-dpu/tf_yolov3_voc.xmodel"
XMODEL = XMODEL_LOCAL if os.path.exists(XMODEL_LOCAL) else XMODEL_PYNQ

def read_power_mw():
    with open(POWER_SENSOR) as f:
        return int(f.read().strip()) / 1000

print(f"Using xmodel: {XMODEL}")
print(f"Current board power: {read_power_mw()/1000:.2f} W (idle)")

## Step 1 — Load DPU Overlay

In [ ]:
from pynq_dpu import DpuOverlay
overlay = DpuOverlay("dpu.bit")
print("DPU overlay loaded!")

## Step 2 — Load YOLOv3 Model
Note the output has **3 tensors** (one per detection scale) unlike ResNet50's single output.

In [ ]:
overlay.load_model(XMODEL)
dpu = overlay.runner

in_t  = dpu.get_input_tensors()
out_t = dpu.get_output_tensors()
print(f"Input shape:   {in_t[0].dims}  (416x416 image)")
print(f"Output shapes: {[t.dims for t in out_t]}")
print(f"  -> 13x13 grid: detects large objects")
print(f"  -> 26x26 grid: detects medium objects")
print(f"  -> 52x52 grid: detects small objects")

## Step 3 — Prepare Buffers

In [ ]:
in_d  = [np.zeros(t.dims, dtype=np.int8) for t in in_t]
out_d = [np.zeros(t.dims, dtype=np.int8) for t in out_t]
print("Buffers ready")

## Step 4 — Warmup

In [ ]:
for _ in range(N_WARMUP):
    job = dpu.execute_async(in_d, out_d)
    dpu.wait(job)
print("Warmup done!")

## Step 5 — Benchmark

In [ ]:
power_samples = []
start = time.time()
for i in range(N_BENCHMARK):
    job = dpu.execute_async(in_d, out_d)
    dpu.wait(job)
    if i % 10 == 0:
        power_samples.append(read_power_mw())
elapsed = time.time() - start

avg_power_w = sum(power_samples) / len(power_samples) / 1000
fps = N_BENCHMARK / elapsed
latency_ms = elapsed / N_BENCHMARK * 1000
fps_per_watt = fps / avg_power_w

print("=" * 40)
print("YOLOV3 DPU BENCHMARK RESULTS")
print("=" * 40)
print(f"Platform:    KV260 DPU (B512)")
print(f"FPS:         {fps:.1f}")
print(f"Latency:     {latency_ms:.1f} ms/frame")
print(f"Power:       {avg_power_w:.2f} W")
print(f"FPS/Watt:    {fps_per_watt:.2f}")
print("=" * 40)

del overlay

In [ ]:
import cv2, numpy as np
from pynq_dpu import DpuOverlay

# VOC class names (20 classes this model was trained on)
VOC_CLASSES = ['aeroplane','bicycle','bird','boat','bottle','bus','car','cat',
               'chair','cow','diningtable','dog','horse','motorbike','person',
               'pottedplant','sheep','sofa','train','tvmonitor']

# Load pre-resized fish image (416x416 for YOLOv3)
IMAGE_PATHS = [
    "/root/jupyter_notebooks/dpu_benchmark/test_image_416.jpg",
    "/home/ubuntu/test_image_416.jpg",
]
img_path = next((p for p in IMAGE_PATHS if os.path.exists(p)), None)
img = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Normalize to int8
img_int8 = (img_rgb.astype(np.float32) / 255.0 * 128).clip(-128, 127).astype(np.int8)

# Run inference
overlay4 = DpuOverlay("dpu.bit")
overlay4.load_model(XMODEL)
dpu4 = overlay4.runner
in_t4  = dpu4.get_input_tensors()
out_t4 = dpu4.get_output_tensors()
in_d4  = [np.zeros(t.dims, dtype=np.int8) for t in in_t4]
out_d4 = [np.zeros(t.dims, dtype=np.int8) for t in out_t4]

in_d4[0][0] = img_int8
job = dpu4.execute_async(in_d4, out_d4)
dpu4.wait(job)

# Parse outputs — each grid cell predicts 3 boxes with (x,y,w,h,conf,20 classes)
print(f"Image: fish.jpg (416x416)")
print(f"YOLOv3 output tensors: {[t.dims for t in out_t4]}")
print(f"  13x13 = large object grid, 26x26 = medium, 52x52 = small")

# Find highest confidence predictions across all scales
best_conf = 0
best_class = -1
for out in out_d4:
    arr = out[0].astype(np.float32)  # shape e.g. (13, 13, 75) = 13x13 grid, 3 boxes x 25
    # Each box: [x, y, w, h, conf, class0..class19]
    arr_reshaped = arr.reshape(-1, 25)
    confs = arr_reshaped[:, 4]
    if confs.max() > best_conf:
        best_conf = confs.max()
        best_idx = confs.argmax()
        class_scores = arr_reshaped[best_idx, 5:]
        best_class = class_scores.argmax()

print(f"\nHighest confidence detection:")
print(f"  Class: {VOC_CLASSES[best_class] if best_class >= 0 else 'none'}")
print(f"  Confidence score: {best_conf:.2f}")
print(f"\nNote: YOLOv3 VOC was not trained on 'fish' as a class.")
print(f"VOC classes are: {', '.join(VOC_CLASSES)}")

del overlay4

## Bonus — What Does YOLOv3 Detect? (Real Image Test)
Uses the 416x416 pre-resized fish image. YOLOv3 was trained on VOC dataset (20 classes: person, car, dog, cat, bird, fish, etc.)